In [ ]:
import os
import subprocess
proc = subprocess.Popen(["node", "main.js"], preexec_fn=os.setsid)  # 현재 폴더에 main.js
print(f"실행 중 (PID={proc.pid})")

실행 중 (PID=22060)


In [6]:
# pip install "python-socketio[client]"
import time
import socketio

SIO_URL = "http://localhost:8080"   # MindServer 기본 포트
AGENT   = "andy"
TEXT    = "build a house"

# Mindcraft 빌드/버전에 따라 이벤트명이 다를 수 있어 여러 후보를 시도
EVENT_CANDIDATES = [
    "external_chat",     # 흔한 네이밍
    "externalMessage",   # 카멜케이스 변형
    "send_chat",         # 직관적 네이밍
    "chat"               # 가장 단순 이벤트
]

sio = socketio.Client(logger=False, engineio_logger=False)

@sio.event
def connect():
    print("connected to MindServer")
    payload = {"agent": AGENT, "text": TEXT}
    for ev in EVENT_CANDIDATES:
        try:
            sio.emit(ev, payload)
            print(f"emitted '{ev}' -> {payload}")
            time.sleep(0.2)
        except Exception as e:
            print(f"failed '{ev}': {e}")
    # 필요 시 계속 붙어 있으면서 응답을 받고 싶으면 disconnect 제거
    sio.disconnect()

@sio.event
def connect_error(err):
    print("connect_error:", err)

@sio.event
def disconnect():
    print("disconnected")

sio.connect(SIO_URL, transports=["websocket", "polling"])


connected to MindServer
emitted 'external_chat' -> {'agent': 'andy', 'text': 'build a house'}
emitted 'externalMessage' -> {'agent': 'andy', 'text': 'build a house'}
emitted 'send_chat' -> {'agent': 'andy', 'text': 'build a house'}
emitted 'chat' -> {'agent': 'andy', 'text': 'build a house'}
